# Intraday Liquidity Calculation

This notebook walks through the end-to-end pipeline for monitoring intraday liquidity:
1. Load and inspect payment transaction data
2. Clean and preprocess the data
3. Compute intraday time buckets
4. Calculate the running liquidity position
5. Visualize the intraday liquidity position

## Step 1 — Load Data

In [ ]:
import pandas as pd
import numpy as np

# Simulate intraday payment transaction data
np.random.seed(42)
n = 200

timestamps = pd.date_range('2024-01-15 08:00', periods=n, freq='3min')

transactions = pd.DataFrame({
    'timestamp': timestamps,
    'amount': np.random.choice(
        np.concatenate([np.random.uniform(1_000, 50_000, 180),
                        np.random.uniform(-50_000, -1_000, 20)]),
        size=n,
        replace=False
    )
})

print(f'Loaded {len(transactions)} transactions')
transactions.head()

## Step 2 — Preprocess Data

In [ ]:
# Ensure correct dtypes
transactions['timestamp'] = pd.to_datetime(transactions['timestamp'])
transactions['amount'] = transactions['amount'].astype(float)

# Drop any rows with missing values
transactions.dropna(inplace=True)

print(transactions.dtypes)
transactions.describe()

## Step 3 — Compute Time Buckets

In [ ]:
# Round timestamps down to the nearest 30-minute bucket
transactions['time_bucket'] = transactions['timestamp'].dt.floor('30min')

print('Unique time buckets:', transactions['time_bucket'].nunique())
transactions[['timestamp', 'time_bucket', 'amount']].head(10)

## Step 4 — Calculate Liquidity Position

In [ ]:
# Aggregate net flows per time bucket
liquidity = (
    transactions
    .groupby('time_bucket', as_index=False)['amount']
    .sum()
    .rename(columns={'amount': 'net_flow'})
)

# Compute running (cumulative) liquidity position
liquidity['liquidity_position'] = liquidity['net_flow'].cumsum()

print(f'Liquidity table shape: {liquidity.shape}')
liquidity.head(10)

## Step 5 — Visualization

In [ ]:
import matplotlib.pyplot as plt

plt.figure()

plt.plot(liquidity['time_bucket'], liquidity['liquidity_position'])

plt.title("Intraday Liquidity Position")

plt.xlabel("Time")

plt.ylabel("Liquidity")

plt.show()